In [1]:
# import findspark
import findspark
# initialize findspark with spark directory
findspark.init()
# import pyspark
import pyspark
# create spark contextp
sc = pyspark.SparkContext()
# create spark session
spark = pyspark.sql.SparkSession(sc)

In [2]:
# inspect spark session
spark

In [60]:
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array, array_to_vector
from pyspark.ml.linalg import Vectors
from pyspark.sql.functions import array
from pyspark.sql.types import NumericType
from pyspark.ml.stat import Summarizer

# 1 Adding CEO_CA sentiment

In [72]:
basetable = spark.read.parquet("../Cleaned_data/biweekly_basetable.parquet")
basetable.show()

+------+--------------+---------------+--------------------+--------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+---------+
|ticker|

In [73]:
basetable.count()

587

In [74]:
ceo_ca_dir = "../Data/Sentiment_data/CEO_CA"
KEY_COLS = {"ticker", "biweekly_index", "biweekly_period", "period_start", "period_end"}
SUFFIX = "_ca"

In [75]:
sentiment = (
    spark.read.parquet(ceo_ca_dir + "/biweekly_sentiment_*.parquet")
    .withColumn("source_file", F.input_file_name())
    .withColumn(
        "ticker",
        F.regexp_extract(
            F.col("source_file"),
            r"biweekly_sentiment_([^/\\]+)\.parquet",
            1
        )
    )
    .drop("source_file")
)
sentiment_ca_renamed = sentiment.select(
    *[
        F.col(c).alias(c + SUFFIX) if c not in KEY_COLS else F.col(c)
        for c in sentiment.columns
    ]
)
sentiment_ca_renamed.show()

+--------------+---------------+------------+----------+-------------+-------------------+-------------------+--------------------+--------------------+-------------------+------------------+------+
|biweekly_index|biweekly_period|period_start|period_end|n_comments_ca|       avg_vader_ca|       std_vader_ca|      avg_roberta_ca|   median_roberta_ca|     min_roberta_ca|    max_roberta_ca|ticker|
+--------------+---------------+------------+----------+-------------+-------------------+-------------------+--------------------+--------------------+-------------------+------------------+------+
|             0|             P0|  2024-01-15|2024-01-26|           68|0.16746617647058826| 0.4453718739007339| 0.11449775817420553| 0.08152793347835541|-0.8893684148788452|0.9877437949180603| ESK.V|
|             1|             P1|  2024-01-29|2024-02-09|           30|0.16385333333333338| 0.3365166035857501| 0.18073276827732723| 0.06870905682444572|-0.8759412169456482|0.9627717733383179| ESK.V|
|    

In [77]:
sentiment_ca_renamed.count()

797

In [83]:
sentiment_ca_renamed = sentiment_ca_renamed.drop("period_start", "period_end")
sentiment_ca_renamed.show()

+--------------+---------------+-------------+-------------------+-------------------+--------------------+--------------------+-------------------+------------------+------+
|biweekly_index|biweekly_period|n_comments_ca|       avg_vader_ca|       std_vader_ca|      avg_roberta_ca|   median_roberta_ca|     min_roberta_ca|    max_roberta_ca|ticker|
+--------------+---------------+-------------+-------------------+-------------------+--------------------+--------------------+-------------------+------------------+------+
|             0|             P0|           68|0.16746617647058826| 0.4453718739007339| 0.11449775817420553| 0.08152793347835541|-0.8893684148788452|0.9877437949180603| ESK.V|
|             1|             P1|           30|0.16385333333333338| 0.3365166035857501| 0.18073276827732723| 0.06870905682444572|-0.8759412169456482|0.9627717733383179| ESK.V|
|             2|             P2|           21| 0.2066095238095238|0.43840288592285465|  0.2475886813675364| 0.084485471248626

In [84]:
sentiment_ca_renamed.count()

797

In [85]:
joined = (
    basetable
    .join(sentiment_ca_renamed, on=["biweekly_index", "biweekly_period", "ticker"], how="left")
)
joined.show()

+--------------+---------------+------+--------------------+--------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+---------+---------

In [79]:
joined.count()

587

In [86]:
joined.write.mode("overwrite").parquet("../Cleaned_data/biweekly_basetable_w_sentiment.parquet")

# 2. Adding Company Sentiment

avg_fin_similarity NOT READY YET

In [119]:
basetable = spark.read.parquet("../Cleaned_data/biweekly_basetable_w_sentiment.parquet")
basetable.show()

+--------------+---------------+------+--------------------+--------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+---------+---------

In [120]:
SUFFIX = "_comp"
KEY_COLS = {"ticker", "biweekly_index", "biweekly_period", "period_start", "period_end"}

In [121]:
company_sentiment = spark.read.parquet("../Cleaned_data/basetable_company_sentiment.parquet")
company_sentiment.show()

+--------------+-----------------+---------+-------------------+---------------------+-------------------+--------------------+----------------------+------------------------+--------------------+------------------+----------+----------+------------------+--------------------+-----------------+--------------------+--------------------+------+------------------+-------------------+--------------------+------------------+
|biweekly_index|next_announcement|n_artices|avg_num_words_title|avg_num_words_content| avg_polarity_title|avg_polarity_content|avg_subjectivity_title|avg_subjectivity_content|          avg_ollama|        std_ollama|min_ollama|max_ollama|           avg_FPI|             std_FPI|avg_Financing_sim|   avg_finbert_array|     avg_finbert_vec|ticker|         accum_FPI|  std_avg_FPI_last3|        prev_std_FPI|      prev_avg_FPI|
+--------------+-----------------+---------+-------------------+---------------------+-------------------+--------------------+----------------------+--

In [122]:
company_sentiment_renamed = company_sentiment.select(
    *[
        F.col(c).alias(c + SUFFIX) if c not in KEY_COLS else F.col(c)
        for c in company_sentiment.columns
    ]
)
company_sentiment_renamed.show()

+--------------+----------------------+--------------+------------------------+--------------------------+-----------------------+-------------------------+---------------------------+-----------------------------+--------------------+------------------+---------------+---------------+------------------+--------------------+----------------------+----------------------+--------------------+------+------------------+----------------------+--------------------+------------------+
|biweekly_index|next_announcement_comp|n_artices_comp|avg_num_words_title_comp|avg_num_words_content_comp|avg_polarity_title_comp|avg_polarity_content_comp|avg_subjectivity_title_comp|avg_subjectivity_content_comp|     avg_ollama_comp|   std_ollama_comp|min_ollama_comp|max_ollama_comp|      avg_FPI_comp|        std_FPI_comp|avg_Financing_sim_comp|avg_finbert_array_comp|avg_finbert_vec_comp|ticker|    accum_FPI_comp|std_avg_FPI_last3_comp|   prev_std_FPI_comp| prev_avg_FPI_comp|
+--------------+------------------

In [123]:
company_sentiment_renamed.count()

836

In [124]:
company_sentiment_renamed.drop("next_announcement_comp")

DataFrame[biweekly_index: bigint, n_artices_comp: bigint, avg_num_words_title_comp: double, avg_num_words_content_comp: double, avg_polarity_title_comp: double, avg_polarity_content_comp: double, avg_subjectivity_title_comp: double, avg_subjectivity_content_comp: double, avg_ollama_comp: double, std_ollama_comp: double, min_ollama_comp: string, max_ollama_comp: string, avg_FPI_comp: double, std_FPI_comp: double, avg_Financing_sim_comp: double, avg_finbert_array_comp: array<double>, avg_finbert_vec_comp: vector, ticker: string, accum_FPI_comp: double, std_avg_FPI_last3_comp: double, prev_std_FPI_comp: double, prev_avg_FPI_comp: double]

In [125]:
joined = (
    basetable
    .join(company_sentiment_renamed, on=["biweekly_index", "ticker"], how="left")
)
joined.show()

+--------------+------+---------------+--------------------+--------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+---------+---------

In [126]:
joined.count()

587

In [127]:
joined.write.mode("overwrite").parquet("../Cleaned_data/biweekly_basetable_w_sentiment_2.parquet")

# Adding Financial Times 

In [128]:
SUFFIX = "_ft"
KEY_COLS = {"biweekly_period", "industry"}

In [129]:
fin_times = spark.read.parquet("../Sentiment_Analysis/Financial_times_sentiment.parquet")
fin_times.show()

+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------+-----------------+--------------+--------------------+-------------------+--------------------+--------------------+----------------+
|biweekly_period|            industry|             content|               title|     cleaned_content|       cleaned_title|       content_lemma|         title_lemma|     title_tokenized|   content_tokenized|num_words_title|num_words_content|polarity_title|    polarity_content| subjectivity_title|subjectivity_content|  finbert_embeddings|ollama_sentiment|
+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------+-----------------+--------------+--------------------+------------

In [130]:
useless_cols = ["content", "title", "cleaned_title", "cleaned_content", "content_lemma", "title_lemma", "title_tokenized", "content_tokenized"]
fin_times = fin_times.drop("content", "title", "cleaned_title","cleaned_content", "content_lemma", "title_lemma", "title_tokenized", "content_tokenized")
fin_times.printSchema()

root
 |-- biweekly_period: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- num_words_title: integer (nullable = true)
 |-- num_words_content: integer (nullable = true)
 |-- polarity_title: double (nullable = true)
 |-- polarity_content: double (nullable = true)
 |-- subjectivity_title: double (nullable = true)
 |-- subjectivity_content: double (nullable = true)
 |-- finbert_embeddings: vector (nullable = true)
 |-- ollama_sentiment: string (nullable = true)



In [131]:
fin_times = fin_times.withColumn(
    "ollama_sentiment",
    F.expr("try_cast(ollama_sentiment as double)")
)
fin_times.show()

+---------------+--------------------+---------------+-----------------+--------------+--------------------+-------------------+--------------------+--------------------+----------------+
|biweekly_period|            industry|num_words_title|num_words_content|polarity_title|    polarity_content| subjectivity_title|subjectivity_content|  finbert_embeddings|ollama_sentiment|
+---------------+--------------------+---------------+-----------------+--------------+--------------------+-------------------+--------------------+--------------------+----------------+
|            P45|              Silver|              7|              292|           0.0| 0.10248968086177393|                0.0| 0.45226719017416694|[0.01531461440026...|            0.85|
|            P45|                Gold|              7|              292|           0.0| 0.10248968086177393|                0.0| 0.45226719017416694|[0.01531461440026...|            0.85|
|            P38|              Silver|              4|      

In [132]:
fin_times.printSchema()

root
 |-- biweekly_period: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- num_words_title: integer (nullable = true)
 |-- num_words_content: integer (nullable = true)
 |-- polarity_title: double (nullable = true)
 |-- polarity_content: double (nullable = true)
 |-- subjectivity_title: double (nullable = true)
 |-- subjectivity_content: double (nullable = true)
 |-- finbert_embeddings: vector (nullable = true)
 |-- ollama_sentiment: double (nullable = true)



Aggregation of variables still need to happen

In [133]:
agg_ft_scalars = (
    fin_times
    .groupBy("industry", "biweekly_period")
    .agg(
        F.count("*").alias("article_count_ft"),
        F.sum("num_words_content").alias("total_words_ft"),
        F.avg("num_words_content").alias("avg_words_ft"),

        # weighted polarity
        (F.sum(F.col("polarity_content") * F.col("num_words_content")) /
         F.sum("num_words_content")).alias("polarity_ft"),

        # weighted subjectivity
        (F.sum(F.col("subjectivity_content") * F.col("num_words_content")) /
         F.sum("num_words_content")).alias("subjectivity_ft"),

        # weighted LLM sentiment
        (F.sum(F.col("ollama_sentiment") * F.col("num_words_content")) /
         F.sum("num_words_content")).alias("ollama_sentiment_ft"),
    )
)

average the embeddings

In [134]:
ft_emb = fin_times.withColumn(
    "embedding_arr",
    vector_to_array("finbert_embeddings")
)

embedding_dim = len(ft_emb.select("embedding_arr").first()[0])

agg_embedding = (
    ft_emb
    .groupBy("industry", "biweekly_period")
    .agg(
        *[
            (
                F.sum(F.col("embedding_arr")[i] * F.col("num_words_content")) /
                F.sum("num_words_content")
            ).alias(f"emb_{i}")
            for i in range(embedding_dim)
        ]
    )
)


agg_embedding = (
    agg_embedding
    .withColumn(
        "finbert_embedding_arr_ft",
        F.array(*[F.col(f"emb_{i}") for i in range(embedding_dim)])
    )
    .drop(*[f"emb_{i}" for i in range(embedding_dim)])
    .withColumn("finbert_embedding_ft", array_to_vector("finbert_embedding_arr_ft"))
    .drop("finbert_embedding_arr_ft")
)
agg_embedding = agg_embedding.orderBy(["industry", "biweekly_period"])


In [135]:
agg_embedding.show()

+--------+---------------+--------------------+
|industry|biweekly_period|finbert_embedding_ft|
+--------+---------------+--------------------+
|  Copper|            P12|[0.08853341639041...|
|  Copper|            P13|[-0.0384738333523...|
|  Copper|            P16|[0.22285652160644...|
|  Copper|            P23|[0.03581451624631...|
|  Copper|            P25|[0.22625121474266...|
|  Copper|            P26|[-0.0140473563224...|
|  Copper|            P27|[0.04405492544174...|
|  Copper|            P28|[0.17380355611031...|
|  Copper|            P29|[0.07515509426593...|
|  Copper|            P30|[0.12590713424377...|
|  Copper|            P31|[0.10656258608924...|
|  Copper|            P33|[0.13651713013163...|
|  Copper|            P34|[0.15336445770874...|
|  Copper|            P35|[0.05827431510324...|
|  Copper|            P36|[0.04207586497068...|
|  Copper|            P38|[0.10339357535607...|
|  Copper|            P40|[0.13025221228599...|
|  Copper|            P41|[0.15521256238

In [136]:
agg_ft_scalars.count()

101

In [137]:
ft_biweekly = (
    agg_ft_scalars
    .join(agg_embedding, ["industry", "biweekly_period"], "left")
)
ft_biweekly.show()

+--------------------+---------------+----------------+--------------+-----------------+--------------------+-------------------+--------------------+--------------------+
|            industry|biweekly_period|article_count_ft|total_words_ft|     avg_words_ft|         polarity_ft|    subjectivity_ft| ollama_sentiment_ft|finbert_embedding_ft|
+--------------------+---------------+----------------+--------------+-----------------+--------------------+-------------------+--------------------+--------------------+
|                Gold|            P14|               1|           350|            350.0|  0.0973939393939394| 0.4193659673659673|                0.65|[0.03255259990692...|
|              Copper|            P16|               1|           555|            555.0| 0.04864543544690604| 0.3930647334691453|                0.85|[0.22285652160644...|
|                Gold|            P11|               1|           654|            654.0| 0.16221171536796544| 0.4878674242424242|           

In [138]:
ft_biweekly.count()

101

In [139]:
basetable_1 = spark.read.parquet("../Cleaned_data/biweekly_basetable_w_sentiment_2.parquet")
basetable_1.show()

+--------------+------+---------------+--------------------+--------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+---------+---------

In [109]:
ft_biweekly.printSchema()

root
 |-- industry: string (nullable = true)
 |-- biweekly_period: string (nullable = true)
 |-- article_count_ft: long (nullable = false)
 |-- total_words_ft: long (nullable = true)
 |-- avg_words_ft: double (nullable = true)
 |-- polarity_ft: double (nullable = true)
 |-- subjectivity_ft: double (nullable = true)
 |-- ollama_sentiment_ft: double (nullable = true)
 |-- finbert_embedding_ft: vector (nullable = true)



In [140]:
joined = (
    basetable_1
    .join(ft_biweekly, on=["biweekly_period", "industry"], how="left")
)
joined.show()

+---------------+--------+--------------+------+--------------------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+---------+---------

In [141]:
joined.count()

587

In [142]:
# get all numeric columns
numeric_cols = [c for c, t in joined.dtypes if isinstance(joined.schema[c].dataType, NumericType)]

# count null values in each column
null_counts = joined.select([F.count(F.when(F.col(c).isNull() | F.isnan(c), c)).alias(c) for c in numeric_cols])
null_counts.show()

+--------------+-----------+------------+-----------+----------+---------+-------------+-----------------------+----+----------------+-----------------+------------------+------------------+------------+----------------+----------------+-----------------+------------------+--------------------+---------+------------+-------+-------+---------+---------------------+------------------------+-----------------------+---------+-------------------------+-------------+------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+-------------+-----------------+-------------+-------------+------------+------------+--------------+-----------------+--------------+--------------+----------------------+--------------+------------------------+--------------------------+----------------

In [143]:
joined.write.mode("overwrite").parquet("../Cleaned_data/biweekly_basetable_w_sentiment_3.parquet")

# Adding De Tijd

In [144]:
detijd = spark.read.parquet("../Sentiment_Analysis/detijd_sentiment.parquet")
detijd.show()

+----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------+-----------------+-----------+---------------+--------------+--------------------+------------------+--------------------+----------------+
|            date|             content|               title|            category|       content_lemma|         title_lemma|     title_tokenized|   content_tokenized|num_words_title|num_words_content|parsed_date|biweekly_period|polarity_title|    polarity_content|subjectivity_title|subjectivity_content|ollama_sentiment|
+----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------+-----------------+-----------+---------------+--------------+--------------------+------------------+--------------------+----------------+
| 18 januari 2024|de europese beurz..

In [145]:
SUFFIX = "_tijd"
KEY_COLS = {"biweekly_period", "industry"}

In [146]:
useless_cols = ["date", "content", "title", "cleaned_title", "cleaned_content", "content_lemma", "title_lemma", "title_tokenized", "content_tokenized"]
detijd = detijd.drop("date", "content", "title", "cleaned_title", "cleaned_content", "content_lemma", "title_lemma", "title_tokenized", "content_tokenized")
detijd.printSchema()

root
 |-- category: string (nullable = true)
 |-- num_words_title: integer (nullable = true)
 |-- num_words_content: integer (nullable = true)
 |-- parsed_date: date (nullable = true)
 |-- biweekly_period: string (nullable = true)
 |-- polarity_title: double (nullable = true)
 |-- polarity_content: double (nullable = true)
 |-- subjectivity_title: double (nullable = true)
 |-- subjectivity_content: double (nullable = true)
 |-- ollama_sentiment: string (nullable = true)



In [147]:
detijd = detijd.withColumnRenamed("category","industry")
detijd = detijd.withColumn(
    "ollama_sentiment",
    F.expr("try_cast(ollama_sentiment as double)")
)
detijd.show()

+--------------------+---------------+-----------------+-----------+---------------+--------------+--------------------+------------------+--------------------+----------------+
|            industry|num_words_title|num_words_content|parsed_date|biweekly_period|polarity_title|    polarity_content|subjectivity_title|subjectivity_content|ollama_sentiment|
+--------------------+---------------+-----------------+-----------+---------------+--------------+--------------------+------------------+--------------------+----------------+
|              Copper|              4|              108| 2024-01-18|             P0|           0.0|                 0.0|               0.0|                 0.0|            0.65|
|              Copper|              5|              686| 2024-10-30|            P20|           0.0| 0.06818181818181818|               0.0| 0.22727272727272727|            0.72|
|              Copper|              5|             1039| 2024-09-06|            P16|           0.0|0.039999999

aggregate to biweekly

In [148]:
agg_tijd_scalars = (
    detijd
    .groupBy("industry", "biweekly_period")
    .agg(
        F.count("*").alias("article_count_tijd"),
        F.sum("num_words_content").alias("total_words_tijd"),
        F.avg("num_words_content").alias("avg_words_tijd"),

        # weighted polarity
        (F.sum(F.col("polarity_content") * F.col("num_words_content")) /
         F.sum("num_words_content")).alias("polarity_tijd"),

        # weighted subjectivity
        (F.sum(F.col("subjectivity_content") * F.col("num_words_content")) /
         F.sum("num_words_content")).alias("subjectivity_tijd"),

        # weighted LLM sentiment
        (F.sum(F.col("ollama_sentiment") * F.col("num_words_content")) /
         F.sum("num_words_content")).alias("ollama_sentiment_tijd"),
    )
)

In [149]:
agg_tijd_scalars.show()

+--------------------+---------------+------------------+----------------+------------------+--------------------+-------------------+---------------------+
|            industry|biweekly_period|article_count_tijd|total_words_tijd|    avg_words_tijd|       polarity_tijd|  subjectivity_tijd|ollama_sentiment_tijd|
+--------------------+---------------+------------------+----------------+------------------+--------------------+-------------------+---------------------+
|                Gold|            P14|                28|           23204| 828.7142857142857| 0.07587226176395878|0.23017585904444043|     0.38613040855025|
|              Copper|            P16|                20|           10171|            508.55| 0.04677256699046591|0.28289624459973983|   0.3435434077278538|
|              Copper|           NULL|                78|           50334| 645.3076923076923| 0.03623221837388365|0.23061238787052962|   0.3777736718718958|
|              Copper|            P14|                 6| 

In [150]:
basetable_1 = spark.read.parquet("../Cleaned_data/biweekly_basetable_w_sentiment_3.parquet")
basetable_1.show()

+---------------+--------+--------------+------+--------------------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+---------+---------

In [151]:
joined = (
    basetable_1
    .join(agg_tijd_scalars, on=["biweekly_period", "industry"], how="left")
)
joined.show()

+---------------+--------+--------------+------+--------------------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+---------+---------

In [152]:
joined.count()

587

In [153]:
joined.write.mode("overwrite").parquet("../Cleaned_data/biweekly_basetable_w_sentiment_4.parquet")

# Checking And Impurting Missing Values

### Financial Times

In [154]:
industry_coverage = (
    joined
    .groupBy("industry")
    .agg(
        F.count("*").alias("total_rows"),
        F.sum(F.when(F.col("article_count_ft").isNotNull(), 1).otherwise(0))
            .alias("rows_with_ft")
    )
    .withColumn("coverage_ratio", F.col("rows_with_ft") / F.col("total_rows"))
    .orderBy("coverage_ratio")
)

industry_coverage.show(50, truncate=False)

+--------------------------------+----------+------------+-------------------+
|industry                        |total_rows|rows_with_ft|coverage_ratio     |
+--------------------------------+----------+------------+-------------------+
|Other Industrial Metals & Mining|206       |79          |0.38349514563106796|
|Other Precious Metals & Mining  |90        |39          |0.43333333333333335|
|Silver                          |54        |28          |0.5185185185185185 |
|Copper                          |70        |38          |0.5428571428571428 |
|Gold                            |167       |152         |0.9101796407185628 |
+--------------------------------+----------+------------+-------------------+



In [155]:
joined = joined.withColumn(
    "ft_coverage",
    F.when(F.col("article_count_ft") > 0, 1).otherwise(0)
)

filling volume metrics with zero

In [157]:
joined = joined.fillna({
    "article_count_ft": 0,
    "total_words_ft": 0,
    "avg_words_ft": 0
})

filling with median

In [158]:
industry_medians = (
    joined
    .groupBy("industry")
    .agg(
        F.expr("percentile_approx(polarity_ft, 0.5)").alias("polarity_med"),
        F.expr("percentile_approx(subjectivity_ft, 0.5)").alias("subjectivity_med"),
        F.expr("percentile_approx(ollama_sentiment_ft, 0.5)").alias("ollama_med")
    )
)

joined = (
    joined
    .join(industry_medians, "industry", "left")
    .withColumn(
        "polarity_ft",
        F.when(F.col("polarity_ft").isNull(), F.col("polarity_med"))
         .otherwise(F.col("polarity_ft"))
    )
    .withColumn(
        "subjectivity_ft",
        F.when(F.col("subjectivity_ft").isNull(), F.col("subjectivity_med"))
         .otherwise(F.col("subjectivity_ft"))
    )
    .withColumn(
        "ollama_sentiment_ft",
        F.when(F.col("ollama_sentiment_ft").isNull(), F.col("ollama_med"))
         .otherwise(F.col("ollama_sentiment_ft"))
    )
    .drop("polarity_med", "subjectivity_med", "ollama_med")
)

Vector imputing

In [160]:
industry_emb_centroid = (
    joined
    .where(F.col("finbert_embedding_ft").isNotNull())
    .groupBy("industry")
    .agg(
        Summarizer.mean(F.col("finbert_embedding_ft"))
        .alias("emb_centroid_ft")
    )
)

In [161]:
joined = (
    joined
    .join(industry_emb_centroid, "industry", "left")
    .withColumn(
        "finbert_embedding_ft",
        F.when(
            F.col("finbert_embedding_ft").isNull(),
            F.col("emb_centroid_ft")
        ).otherwise(F.col("finbert_embedding_ft"))
    )
    .drop("emb_centroid_ft")
)

In [162]:
joined.show(20)

+--------+---------------+--------------+------+--------------------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+---------+---------

### De Tijd

In [164]:
industry_coverage = (
    joined
    .groupBy("industry")
    .agg(
        F.count("*").alias("total_rows"),
        F.sum(F.when(F.col("article_count_tijd").isNotNull(), 1).otherwise(0))
            .alias("rows_with_tijd")
    )
    .withColumn("coverage_ratio", F.col("rows_with_tijd") / F.col("total_rows"))
    .orderBy("coverage_ratio")
)

industry_coverage.show(50, truncate=False)

+--------------------------------+----------+--------------+------------------+
|industry                        |total_rows|rows_with_tijd|coverage_ratio    |
+--------------------------------+----------+--------------+------------------+
|Other Industrial Metals & Mining|206       |134           |0.6504854368932039|
|Other Precious Metals & Mining  |90        |70            |0.7777777777777778|
|Silver                          |54        |46            |0.8518518518518519|
|Copper                          |70        |70            |1.0               |
|Gold                            |167       |167           |1.0               |
+--------------------------------+----------+--------------+------------------+



In [165]:
joined = joined.withColumn(
    "tijd_coverage",
    F.when(F.col("article_count_tijd") > 0, 1).otherwise(0)
)

In [166]:
joined = joined.fillna({
    "article_count_tijd": 0,
    "total_words_tijd": 0,
    "avg_words_tijd": 0
})

In [167]:
industry_medians = (
    joined
    .groupBy("industry")
    .agg(
        F.expr("percentile_approx(polarity_tijd, 0.5)").alias("polarity_med"),
        F.expr("percentile_approx(subjectivity_tijd, 0.5)").alias("subjectivity_med"),
        F.expr("percentile_approx(ollama_sentiment_tijd, 0.5)").alias("ollama_med")
    )
)

joined = (
    joined
    .join(industry_medians, "industry", "left")
    .withColumn(
        "polarity_tijd",
        F.when(F.col("polarity_tijd").isNull(), F.col("polarity_med"))
         .otherwise(F.col("polarity_tijd"))
    )
    .withColumn(
        "subjectivity_tijd",
        F.when(F.col("subjectivity_tijd").isNull(), F.col("subjectivity_med"))
         .otherwise(F.col("subjectivity_tijd"))
    )
    .withColumn(
        "ollama_sentiment_tijd",
        F.when(F.col("ollama_sentiment_tijd").isNull(), F.col("ollama_med"))
         .otherwise(F.col("ollama_sentiment_tijd"))
    )
    .drop("polarity_med", "subjectivity_med", "ollama_med")
)

In [168]:
joined.show()

+--------+---------------+--------------+------+--------------------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+---------+---------

In [169]:
joined.count()

587

In [170]:
joined.write.mode("overwrite").parquet("../Cleaned_data/biweekly_basetable_w_sentiment.parquet")